# Exploratory Data Analysis (EDA)
This notebook performs EDA on the cleaned DrugComb dataset.
**Objectives:**
- Analyze synergy score distributions.
- Visualize drug and cell line coverage.
- Check for missing values.
- Verify SMILES validity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path
sys.path.append(os.path.abspath(os.path.join('../src')))

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Paths
RAW_DIR = '../data/raw'
DATA_DIR = '../data/processed'
CLEANED_DATA_PATH = os.path.join(DATA_DIR, 'cleaned_drugcomb.csv')
FEATURES_PATH = os.path.join(DATA_DIR, 'features_sample.csv')

In [ ]:
# Load cleaned dataset
df = pd.read_csv(CLEANED_DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Synergy Score Distributions
synergy_metrics = ['ZIP', 'Bliss', 'Loewe', 'HSA']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, metric in enumerate(synergy_metrics):
    sns.histplot(df[metric], kde=True, ax=axes[i], bins=50)
    axes[i].set_title(f'{metric} Score Distribution')
    axes[i].set_xlabel('Synergy Score')
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Summary statistics
df[synergy_metrics].describe()

## Investigating the wierd values

In [ ]:
df[df['ZIP'] < -1e2]

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'DrugComb/drugcombs_scored.csv'))

In [ ]:
df_raw[df_raw['ZIP'] < -1e2]

Let's just **remove** these columns

In [ ]:
df = df[(df['ZIP'] > -1e2) & (df['ZIP'] < 100)]

In [ ]:
# Synergy Score Distributions
synergy_metrics = ['ZIP', 'Bliss', 'Loewe', 'HSA']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, metric in enumerate(synergy_metrics):
    sns.histplot(df[metric], kde=True, ax=axes[i], bins=50)
    axes[i].set_title(f'{metric} Score Distribution')
    axes[i].set_xlabel('Synergy Score')
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Summary statistics
df[synergy_metrics].describe()

In [ ]:
# Drug and Cell Line Counts
n_drugs = len(set(df['Drug1'].unique()) | set(df['Drug2'].unique()))
n_cell_lines = df['Cell line'].nunique()

print(f"Total unique drugs: {n_drugs}")
print(f"Total unique cell lines: {n_cell_lines}")

# Top 20 most frequent drugs (Drug1 + Drug2)
drug_counts = pd.concat([df['Drug1'], df['Drug2']]).value_counts().head(20)

plt.figure(figsize=(14, 6))
sns.barplot(x=drug_counts.index, y=drug_counts.values)
plt.title('Top 20 Most Frequent Drugs')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.show()

# Top 20 most frequent cell lines
cell_counts = df['Cell line'].value_counts().head(20)

plt.figure(figsize=(14, 6))
sns.barplot(x=cell_counts.index, y=cell_counts.values)
plt.title('Top 20 Most Frequent Cell Lines')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.show()

In [ ]:
# Missingness Analysis
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

print("Missing values count per column:")
print(df.isnull().sum())

In [ ]:
# Feature Analysis (Sample)
if os.path.exists(FEATURES_PATH):
    df_features = pd.read_csv(FEATURES_PATH)
    print(f"Loaded sample features shape: {df_features.shape}")
    
    # Select only numeric columns for correlation (exclude IDs and SMILES)
    numeric_cols = df_features.select_dtypes(include=[np.number]).columns
    
    # Calculate correlation between some descriptors and synergy scores
    # We'll pick a few key descriptors
    key_descriptors = ['drug1_MolWt', 'drug1_MolLogP', 'drug1_TPSA', 
                       'drug2_MolWt', 'drug2_MolLogP', 'drug2_TPSA',
                       'ZIP', 'Bliss', 'Loewe', 'HSA']
    
    # Filter for columns that actually exist
    available_cols = [c for c in key_descriptors if c in df_features.columns]
    
    if available_cols:
        corr = df_features[available_cols].corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
        plt.title('Correlation: Descriptors vs Synergy')
        plt.show()
    else:
        print("Key descriptors not found in feature file.")
else:
    print("Features sample file not found. Run Day 6 feature engineering first.")

# Lower Dimensional Analysis

since we see high correlation between all the metrics only except loewe, we plot ZIP and Loewe to save compute

In [ ]:
df_dataset = pd.read_csv(DATA_DIR + "/features_sample.csv")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


def plot_dim_reduction(df, label_col, feature_cols=None, n_components=2,
                       random_state=42, cmap="viridis"):
    """
    Plots PCA, t-SNE, and UMAP for the given dataframe and label column.

    Args:
        df: pandas DataFrame containing the data.
        label_col: str, column name to color points by.
        feature_cols: list of str, columns to use as features. If None, uses
            all numeric columns except label_col.
        n_components: int, number of dimensions for reduction (default 2).
        random_state: int, random seed for reproducibility.
        cmap: str or Colormap, matplotlib colormap for continuous labels.
    """
    # Select features
    if feature_cols is None:
        feature_cols = df.select_dtypes(include="number").columns.tolist()
        if label_col in feature_cols:
            feature_cols.remove(label_col)

    X = df[feature_cols].values
    y_raw = df[label_col].values

    # Decide if label is continuous or categorical
    is_numeric = pd.api.types.is_numeric_dtype(df[label_col])
    if is_numeric and df[label_col].nunique() > 20:
        label_type = "continuous"
    else:
        label_type = "categorical"

    # Dimensionality reduction
    pca = PCA(n_components=n_components, random_state=random_state)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=n_components, random_state=random_state)
    X_tsne = tsne.fit_transform(X)

    reducer = umap.UMAP(n_components=n_components, random_state=random_state)
    X_umap = reducer.fit_transform(X)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if label_type == "continuous":
        # Use matplotlib scatter to easily add colorbars
        for ax, X_emb, title in zip(
            axes,
            [X_pca, X_tsne, X_umap],
            ["PCA", "t-SNE", "UMAP"],
        ):
            sc = ax.scatter(
                X_emb[:, 0],
                X_emb[:, 1],
                c=y_raw,
                cmap=cmap,
                s=20,
                alpha=0.8,
            )
            ax.set_title(title)
            cb = plt.colorbar(sc, ax=ax)
            cb.set_label(label_col)
    else:
        # Treat label as categorical, use seaborn with legend
        y_cat = df[label_col].astype("category")
        for ax, X_emb, title in zip(
            axes,
            [X_pca, X_tsne, X_umap],
            ["PCA", "t-SNE", "UMAP"],
        ):
            sns.scatterplot(
                x=X_emb[:, 0],
                y=X_emb[:, 1],
                hue=y_cat,
                ax=ax,
                palette="tab10",
                legend="full",
                s=20,
            )
            ax.set_title(title)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="center right", title=label_col)
        for ax in axes:
            ax.get_legend().remove()

    plt.tight_layout()
    plt.show()


In [ ]:
plot_dim_reduction(df_dataset, label_col='ZIP')

## Further analysis with MolBERT